## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

In [4]:
!pip install pypdf gradio

   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
    --------------------------------------- 0.3/19.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.7 MB 1.7 MB/s eta 0:00:12
   -- ------------------------------------- 1.0/19.7 MB 1.8 MB/s eta 0:00:11
   -- ------------------------------------- 1.3/19.7 MB 1.9 MB/s eta 0:00:10
   --- ------------------------------------ 1.8/19.7 MB 1.9 MB/s eta 0:00:10
   ---- ----------------------------------- 2.4/19.7 MB 2.0 MB/s eta 0:00:09
   ----- ---------------------------------- 2.6/19.7 MB 1.9 MB/s eta 0:00:09
   ------ --------------------------------- 3.1/19.7 MB 2.0 MB/s eta 0:00:09
   ------ --------------------------------- 3.4/19.7 MB 2.0 MB/s eta 0:00:09
   ------- -------------------------------- 3.9/19.7 MB 2.0 MB/s eta 0:00:08
   --------- ------------------------------ 4.5/19.7 MB 2.1 MB/s eta 0:00:08
   ---------- ----------------------------- 5.0/19.7 MB 2.0 MB/s eta 0:00:08
   ----------

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)

True

In [3]:
reader = PdfReader("me/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
Noida
mayankbtp108@gmail.com
www.linkedin.com/in/
mayank1008agarwal (LinkedIn)
Top Skills
PostgreSQL
Web Development
Java
Certifications
Java Full Stack
Unity Beginner to Advance : Game
Development
Data structures and algorithmn
HackXtreme
Java Data Structures and Algorithm
Masterclass
Honors-Awards
Selected as a Top-10 in Grand
Finale
Mayank Agarwal
Gen AI & Backend Engineer | Designing scalable microservices
(Spring Boot, Kotlin) | Passionate about AI systems & game
development | Linux
Noida, Uttar Pradesh, India
Summary
Hi, I'm Mayank Agarwal—a B.Tech CSE graduate and passionate
full-stack developer currently working as an SDE Apprentice at S&P
Global Inc.
Skilled in building scalable microservices with Spring Boot, following
clean architecture and REST and GraphQL principles. Proficient with
Git for version control, efficient branching, and team collaboration to
ensure smooth development workflows.
Currently skilling myself in machine learning along with the
integration

In [5]:
import os
import requests

In [6]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")

In [7]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [8]:
name = "Mayank Agarwal"

<h4>Now we are setting up here two diff prompts.One for the system and one for the user</h4>

In [9]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [10]:
system_prompt

"You are acting as Mayank Agarwal. You are answering questions on Mayank Agarwal's website, particularly questions related to Mayank Agarwal's career, background, skills and experience. Your responsibility is to represent Mayank Agarwal for interactions on the website as faithfully as possible. You are given a summary of Mayank Agarwal's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Mayank Agarwal. I'm a software engineer. I'm originally from Bharatpur(Raj.).\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\nNoida\nmayankbtp108@gmail.com\nwww.linkedin.com/in/\nmayank1008agarwal (LinkedIn)\nTop Skills\nPostgreSQL\nWeb Development\nJava\nCertifications\nJava Full Stack\nUnity Beginner to Advance : Game\nDevelopment\nData structures and algorithmn\nHackXtreme\nJava Data Structures and Algorith

In [12]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":OLLAMA_MODEL,
            "messages":messages,
            "stream": False
        }
    )
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [13]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [23]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [24]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [26]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [27]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [30]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [35]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()